# Strands Agents 1 - Foundations

From zero to a working agent that reasons, calls your tools, remembers a conversation, returns typed data, and streams. This is the bedrock for everything later.

**Series:** these notebooks go scratch -> foundations -> intermediate. Run them in order, top to bottom. Every cell here is real, runnable code (verified against `strands-agents` 1.42).

## 0 · What you need (one-time)

1. **Python 3.10+**.
2. An **AWS account** with **Amazon Bedrock model access** enabled for the Claude models we use. In the AWS console: *Bedrock -> Model access -> Enable* **Claude Haiku 4.5** (and optionally **Claude Sonnet 4**). Access is per-region.
3. **AWS credentials** with permission to call Bedrock (`bedrock:InvokeModel` and `bedrock:InvokeModelWithResponseStream`).

You will set the region and credentials two cells down. Nothing else to install by hand - the next cell does it.

### Install the SDK

`%pip` installs into the *kernel that is running this notebook*, so it works the same in Google Colab and in a local VS Code / Jupyter venv.

In [1]:
%pip install -q strands-agents strands-agents-tools
print("installed")

Note: you may need to restart the kernel to use updated packages.
installed


### Configure region + credentials

This cell works in **both** environments:

- **Google Colab:** open the **Secrets** panel (the key icon in the left sidebar), add `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` (and `AWS_SESSION_TOKEN` if you use temporary keys), then run the cell.
- **VS Code / local:** you usually do not need to paste keys here at all - if you have already run `aws configure` or exported `AWS_` variables in your shell, Strands picks them up automatically. The cell just sets your region.

> Set `AWS_DEFAULT_REGION` to the region where you enabled model access.

In [3]:
import os
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"   # <-- match your enabled-model region

try:
    from google.colab import userdata           # only exists on Colab
    os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
    # os.environ["AWS_SESSION_TOKEN"]   = userdata.get("AWS_SESSION_TOKEN")  # if temporary creds
    print("Colab: credentials loaded from Secrets.")
except ModuleNotFoundError:
    print("Local: using your existing AWS credentials (aws configure / env vars).")
print("Region:", os.environ["AWS_DEFAULT_REGION"])

Local: using your existing AWS credentials (aws configure / env vars).
Region: us-east-1


### Choose a model

Strands talks to Bedrock through a `BedrockModel`. Two things to notice:

- **The `us.` prefix.** Newer Claude models are served on-demand only through a *cross-region inference profile*. The id therefore starts with `us.` (or `global.` / `eu.`). Plain `anthropic.claude-...` would raise *"on-demand throughput isn't supported"*.
- **We set timeouts + adaptive retries** on the underlying boto client. The `adaptive` retry mode automatically backs off and retries when Bedrock throttles you - the single most common error under classroom / burst load.

We default to **Haiku 4.5** because it is fast and cheap, which is ideal while learning. Swap in **Sonnet 4** for harder reasoning by changing one argument.

In [4]:
from strands import Agent, tool
from strands.models import BedrockModel
from botocore.config import Config

MODEL_HAIKU  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # fast + cheap (default)
MODEL_SONNET = "us.anthropic.claude-sonnet-4-20250514-v1:0"    # stronger reasoning

def get_model(model_id=MODEL_HAIKU, temperature=0.3, max_tokens=2048):
    """Build a configured Bedrock model. Reuse this everywhere."""
    return BedrockModel(
        model_id=model_id,
        temperature=temperature,        # 0 = deterministic, higher = more varied
        max_tokens=max_tokens,          # cap on the model's reply length
        boto_client_config=Config(
            read_timeout=900, connect_timeout=900,
            retries={"max_attempts": 4, "mode": "adaptive"},   # auto-handle throttling
        ),
    )

MODEL = get_model()

# Small helper: pull clean text out of an AgentResult.
# result.message looks like {"role": "assistant", "content": [{"text": "..."}]}
def say(result):
    return "".join(block.get("text", "") for block in result.message["content"]).strip()

print("Model ready:", MODEL.get_config()["model_id"])

Model ready: us.anthropic.claude-haiku-4-5-20251001-v1:0


### Smoke test (your first real Bedrock call)

If your access, credentials, and region are correct, this prints **setup OK**. If it errors, read the message:

- *AccessDenied / could not load credentials* -> credentials/region not set (cell above).
- *on-demand throughput isn't supported* or *model id ... not found* -> your account exposes this model under a different inference-profile prefix; try `global.` instead of `us.`, or run `aws bedrock list-inference-profiles` to see what is available.

In [5]:
smoke = Agent(model=MODEL, callback_handler=None)   # callback_handler=None => no streaming print
print(say(smoke("Reply with exactly: setup OK")))

setup OK


---
## 1 · Your first agent

An agent is three things: a **model**, an optional **system prompt** (its role), and a list of **tools**. With no tools, it is just a well-configured chat call. Calling `agent("...")` runs the agent loop and returns an `AgentResult`.

In [6]:
agent = Agent(model=MODEL, callback_handler=None)

result = agent("In one sentence, what is an AI agent?")
print(say(result))

An AI agent is a software system that perceives its environment, makes decisions based on that information, and takes actions to achieve specific goals.


## 2 · What came back: the result and its metrics

`agent(...)` returns an `AgentResult`. Two useful parts:

- `result.message` - the reply, as a dict with a `role` and a list of `content` blocks. Our `say()` helper flattens the text.
- `result.metrics.get_summary()` - an audit trail: how many loop cycles ran, tokens used, latency, and per-tool stats. This is how you understand cost and behavior.

In [12]:
result = agent("What is 17 * 23?")
print("TEXT :", say(result))

summary = result.metrics.get_summary()
print("cycles      :", summary["total_cycles"])
print("tokens      :", summary["accumulated_usage"]["totalTokens"])
print("latency ms  :", summary["accumulated_metrics"]["latencyMs"])

print("Model config:", agent.model.get_config())
print(result.metrics.get_summary())

TEXT : 17 * 23 = 391
cycles      : 7
tokens      : 858
latency ms  : 6684
Model config: {'model_id': 'us.anthropic.claude-haiku-4-5-20251001-v1:0', 'include_tool_result_status': 'auto', 'temperature': 0.3, 'max_tokens': 2048, 'context_window_limit': 200000}
{'total_cycles': 7, 'total_duration': 8.674335956573486, 'average_cycle_time': 1.2391908509390694, 'tool_usage': {}, 'traces': [{'id': '89b43122-f9e1-414f-8d1f-6abaf474c857', 'name': 'Cycle 1', 'raw_name': None, 'parent_id': None, 'start_time': 1780636602.55722, 'end_time': 1780636604.022305, 'duration': 1.4650850296020508, 'children': [{'id': '7b718f30-77dc-466f-8c00-88494284bcd2', 'name': 'stream_messages', 'raw_name': None, 'parent_id': '89b43122-f9e1-414f-8d1f-6abaf474c857', 'start_time': 1780636602.557338, 'end_time': 1780636604.022031, 'duration': 1.4646930694580078, 'children': [], 'metadata': {}, 'message': {'role': 'assistant', 'content': [{'text': 'An AI agent is a software system that perceives its environment, makes deci

## 3 · System prompts: giving the agent a role

The **system prompt** sets behavior and tone for the whole conversation. It is the code equivalent of the *Instructions* box in a console agent builder.

In [13]:
pirate = Agent(
    model=MODEL,
    system_prompt="You are a 17th-century pirate. Answer briefly, in character.",
    callback_handler=None,
)
print(say(pirate("How is the weather looking?")))

*squints at the horizon and sniffs the wind*

Aye, the skies be turnin' gray to the north, and the gulls are flyin' inland—sure signs of a squall brewin'. The barometer in me bones tells me we've got foul weather comin' within the hour, mark me words.

Best we make for shelter or batten down the hatches, lest we find ourselves in a proper tempest. The sea's been too calm of late—she's due for a tantrum.


## 4 · Custom tools with `@tool`

A **tool** is a Python function the model can decide to call. You expose one with the `@tool` decorator.

Three pieces matter, and the model reads all of them:

- **`@tool`** - a *decorator*. Writing `@tool` on the line above a function wraps it so Strands can register it. You do not call it yourself; the agent does.
- **Type hints** (`city: str -> str`) - these become the input/output *schema* the model must fill in correctly.
- **The docstring** - this is the tool's *description*. The model uses it to decide *when* to call the tool, so write it for the model, not for humans.

In [14]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: the city name, e.g. "Bengaluru"
    Returns:
        a short human-readable weather description
    """
    # A real tool would call a weather API. We fake it so this runs anywhere.
    fake = {"bengaluru": "31C and humid", "london": "12C and drizzling"}
    return fake.get(city.lower(), f"No data for {city}, assume mild and clear.")

weather_agent = Agent(model=MODEL, tools=[get_weather], callback_handler=None)
print(say(weather_agent("Should I carry an umbrella in London today?")))

Yes, you should definitely carry an umbrella in London today! It's currently 12°C (about 54°F) and drizzling, so rain protection would be a good idea. The drizzle might continue or develop into heavier rain, so an umbrella will help keep you dry and comfortable.


The agent saw the question, decided `get_weather` was relevant, called it with `city="London"`, read the result, and answered. You never wrote any "if the user asks about weather" routing - the model did that.

## 5 · Built-in tools

The `strands-agents-tools` package ships ready-made tools so you do not rebuild common things. Here are two simple ones.

In [ ]:
from strands_tools import calculator, current_time

helper = Agent(model=MODEL, tools=[calculator, current_time], callback_handler=None)
print(say(helper("What time is it right now, and what is the square root of 4096?")))

╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ sqrt(4096)          │                                                                            │
│  │ Result    │ 64                  │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Current Time:** 2026-06-05T05:24:31.159731+00:00 (UTC)

**Square root of 4096:** 64



## 6 · Several tools, and letting the model choose

Give the agent a few tools and one multi-part request. It will pick the right tool for each part and may call several in one turn. `agent.tool_names` lists what it has.

In [17]:
@tool
def to_celsius(fahrenheit: float) -> float:
    """Convert a temperature from Fahrenheit to Celsius."""
    return round((fahrenheit - 32) * 5 / 9, 1)

multi = Agent(model=MODEL, tools=[get_weather, to_celsius, calculator], callback_handler=None)
print("Tools available:", multi.tool_names)
print()
print(say(multi("What's the weather in Bengaluru, and what is 98.6F in Celsius?")))

Tools available: ['get_weather', 'to_celsius', 'calculator']

Here's the information you requested:

**Weather in Bengaluru:** 31°C and humid

**Temperature Conversion:** 98.6°F = **37.0°C**

(Note: 98.6°F is the normal human body temperature!)


## 7 · Calling a tool directly (no model)

Sometimes you want to run a tool yourself - in a unit test, or for a deterministic step. Use `agent.tool.<name>(...)`. This skips the model entirely and returns the raw tool result.

In [22]:
manual_tool_call_val = multi.tool.to_celsius(fahrenheit=212)

print("Manual tool call result:", manual_tool_call_val)
print(manual_tool_call_val["content"][0]["text"])

Manual tool call result: {'toolUseId': 'tooluse_to_celsius_855316312', 'status': 'success', 'content': [{'text': '100.0'}]}
100.0


The result is a dict with `status` and a `content` list - the same shape the model receives internally. The number is inside `content[0]["text"]`.

## 8 · Conversation memory within a run

An `Agent` keeps a running history in `agent.messages`. Ask a follow-up and it remembers, because the whole thread is sent to the model each turn.

In [24]:
chat = Agent(model=MODEL, callback_handler=None)
chat("My name is Akash and I teach AI bootcamps.")
follow = chat("Given that, suggest a fun demo topic for my class. One line.")
print(say(follow))
print("\nMessages stored so far:", len(chat.messages))
print("\nMessages stored so far:", chat.messages)

Build a chatbot that roasts students based on their bootcamp assignment submissions—funny, engaging, and teaches prompt engineering + API integration in one go.

Messages stored so far: 4

Messages stored so far: [{'role': 'user', 'content': [{'text': 'My name is Akash and I teach AI bootcamps.'}]}, {'role': 'assistant', 'content': [{'text': "# Nice to meet you, Akash!\n\nThat's a great field to be in. AI bootcamps are in high demand right now, and teaching them puts you in a position to shape how people enter this rapidly evolving space.\n\nI'm curious about your work—are you teaching:\n- **Foundational AI concepts** to complete beginners?\n- **Practical skills** (Python, ML frameworks, LLMs)?\n- **Specialized tracks** (NLP, computer vision, etc.)?\n- **A mix** of theory and hands-on projects?\n\nAnd what's been your biggest challenge or win so far in teaching? I'd be happy to help with curriculum ideas, explanations of tricky concepts, or anything else related to your bootcamps."}], 

This memory lives only as long as this `chat` object. Notebook 2 shows how to make it survive restarts, summarize long histories, and store data the model should *not* see.

## 9 · Structured output (typed, validated)

Free text is hard to use in code. `agent.structured_output(Model, prompt)` makes the model fill a **Pydantic** schema and returns a real Python object.

*Pydantic* is a data-validation library. A `BaseModel` subclass defines fields with types; `Field(...)` adds a description that guides the model. If the model's answer does not fit the types, Strands retries.

In [25]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="the film's title")
    year: int = Field(description="release year")
    genres: list[str] = Field(description="up to three genres")

so_agent = Agent(model=MODEL, callback_handler=None)
movie = so_agent.structured_output(Movie, "Describe the movie Inception.")

print(type(movie).__name__)
print("title :", movie.title)
print("year  :", movie.year)          # a real int
print("genres:", movie.genres)        # a real list

/var/folders/0s/szt9vrsx5px2yt75kmpf4_j40000gn/T/ipykernel_11685/3757320490.py:9: DeprecationWarning: Agent.structured_output method is deprecated. You should pass in `structured_output_model` directly into the agent invocation. see: https://strandsagents.com/latest/documentation/docs/user-guide/concepts/agents/structured-output/
  movie = so_agent.structured_output(Movie, "Describe the movie Inception.")


Movie
title : Inception
year  : 2010
genres: ['Science Fiction', 'Thriller', 'Action']


## 10 · Streaming responses

For chat UIs you want tokens as they arrive, not one block at the end. `agent.stream_async(prompt)` is an **async iterator**.

Quick syntax primer:

- **`async def`** defines a coroutine (a function that can pause for I/O).
- **`await`** waits for an async result.
- **`async for ... in ...`** loops over items as they are produced.

Notebooks (Colab and Jupyter) let you `await` at the top level, so we can call the coroutine directly.

In [ ]:
stream_agent = Agent(model=MODEL, callback_handler=None)

async def run_stream():
    async for event in stream_agent.stream_async("Count from 1 to 50 with a word for each."):
        if "data" in event:                 # text chunk events carry a "data" key
            print(event["data"], end="", flush=True)
    print()

await run_stream()   # if this line errors in your runtime, use: import asyncio; asyncio.run(run_stream())

1. One
2. Two
3. Three
4. Four
5. Five


---
## Recap

You can now: build an agent, set its role, give it built-in and custom tools, read its metrics, call tools directly, hold a conversation, get typed output, and stream.

**Next - Notebook 2:** memory that lasts (state, conversation managers, sessions) and connecting external tools over MCP.